# LANGFENS — RQ1-TRN-03A: Dedicated Fold 2 Preflight Verification on Kaggle Tesla T4x2
Official preflight verification notebook for Fold 2 Baseline R (`KAGGLE_T4_CUDA_FP16_AMP`).

## Operational Role & Safety Invariants
- **Role:** Independent, standalone preflight staging and persistence verification.
- **Preflight Only:** Zero training steps, zero optimizer updates, zero model weights generated.
- **Hardware Target:** Tesla T4 × 2 GPU (`Driver: >=580`, `CUDA: >=13.0`, 15,360 MiB VRAM per GPU).
- **Dataset Staging:** Authorized Fold 2 Train+Val package (`minkhoaa/langfens-rq1-fold2-staging`).
- **Sealed Test Isolation:** Active test speakers (`BWC`, `HKK`, `HQTV`, `MBMPS`, `SKA`, `SVBI`) are 100% physically absent.
- **Manifest Integrity:** Verifies split manifest SHA-256 (`7d6bfdf62bc6702b449048968f20e8c78df5b174be341b8d081667982980804b`).
- **Dataset Integrity:** Verifies fold package SHA-256 (`112cc9e0cc5e37e5fa034df504eca019dd6ad438c96cb642335ecf10a802f0d9`).
- **Remote Persistence Probe:** Validates live authenticated write-readback-delete probe to private Hugging Face model repository `minkhoaa/langfens-rq1-cross-fold-checkpoints`.
- **Credential Hygiene:** Resolves `HF_TOKEN` strictly via Kaggle Secrets (`UserSecretsClient`). Zero token files on disk, zero CLI token arguments.
- **Reviewed Training Base Commit:** `cc56b82ca1a9a6ceeca85929e07db4cac8537348`.


In [ ]:
# Phase 1: Hardware & Environment Verification
import subprocess, sys

# Checked execution of nvidia-smi
subprocess.run(["nvidia-smi"], check=True)

import torch
print(f"[GPU] CUDA Available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("FAIL-CLOSED: CUDA GPU is not available on this runtime!")
print(f"[GPU] Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"[GPU {i}]: {torch.cuda.get_device_name(i)}")
assert "Tesla T4" in torch.cuda.get_device_name(0), (
    f"FAIL-CLOSED: Expected Tesla T4 accelerator, got {torch.cuda.get_device_name(0)}"
)

# Verify required core dependencies
import yaml, transformers, huggingface_hub
print(f"[ENV] PyTorch version: {torch.__version__}")
print(f"[ENV] Transformers version: {transformers.__version__}")
print(f"[ENV] Hugging Face Hub version: {huggingface_hub.__version__}")


In [ ]:
# Phase 2: Credentials & Secrets Resolution (Strict Kaggle Secrets Path)
# In accordance with AGENTS.md, tokens are never read from disk files or passed via CLI.
import os

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token and token.strip():
        os.environ["HF_TOKEN"] = token.strip()
        print("[AUTH] Successfully loaded HF_TOKEN from Kaggle Secrets.")
except Exception as e:
    print(f"[AUTH NOTE] Kaggle Secrets UserSecretsClient query: {e}")

if "HF_TOKEN" not in os.environ or not os.environ["HF_TOKEN"].strip():
    raise RuntimeError(
        "FAIL-CLOSED: HF_TOKEN is required in Kaggle Secrets (or environment) for private remote mirroring. "
        "Attach HF_TOKEN under Add-ons -> Secrets in the Kaggle notebook."
    )
print("[AUTH] Verified HF_TOKEN presence in environment for private remote mirroring.")


In [ ]:
# Phase 3: Clone, Checkout Reviewed Training Commit & Verify Tree
import os, subprocess, sys
from pathlib import Path

repo_dir = Path("/kaggle/working/repo")
if not repo_dir.exists():
    print("[GIT] Cloning repository into /kaggle/working/repo...")
    subprocess.run(
        ["git", "clone", "https://github.com/minkhoaa/Langfens-RQ1-Speech-Research.git", str(repo_dir)],
        check=True
    )

os.chdir(str(repo_dir))
subprocess.run(["git", "fetch", "--all"], cwd=repo_dir, check=True)

# Pinned immutable training source commit (closing Step 04C launch-gate corrections)
PINNED_COMMIT_SHA = "cc56b82ca1a9a6ceeca85929e07db4cac8537348"
print(f"[GIT] Checking out pinned training commit: {PINNED_COMMIT_SHA}...")
subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT_SHA], cwd=repo_dir, check=True)

actual_head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True).strip().lower()
print(f"[GIT] Verified Training Commit HEAD: {actual_head}")
assert actual_head == PINNED_COMMIT_SHA.lower(), (
    f"FAIL-CLOSED: Commit SHA mismatch: actual {actual_head} != expected {PINNED_COMMIT_SHA}"
)

status = subprocess.check_output(["git", "status", "--porcelain"], cwd=repo_dir, text=True).strip()
assert len(status) == 0, f"Dirty tree: {status}"
print("[GIT] Repository verified clean at pinned training commit.")

sys.path.insert(0, str(repo_dir / "src"))
sys.path.insert(0, str(repo_dir))


In [ ]:
# Phase 4: Staging Audit & Preflight Verification (verify_fold2)
# Executes preflight staging check, validates split manifest SHA, verifies 0 test speaker
# leakage, checks auto-extracted outer zip reconstruction, and executes live Hugging Face
# write-readback-delete probe. Zero model weights or training steps are executed in this entrypoint.
import subprocess, sys
from pathlib import Path

staging_input = Path("/kaggle/input/langfens-rq1-fold2-staging")
print(f"[INPUT AUDIT] Checking dataset input directory: {staging_input}")
if staging_input.is_dir():
    print(f"[INPUT AUDIT] Directory contents: {[p.name for p in staging_input.iterdir()]}")
else:
    print(f"[INPUT AUDIT WARNING] {staging_input} does not exist yet (requires dataset attachment).")

preflight_cmd = [
    sys.executable,
    "deploy/kaggle/kaggle_entrypoint.py",
    "--mode", "verify_fold2",
    "--commit-sha", PINNED_COMMIT_SHA,
    "--kaggle-input-dir", str(staging_input),
    "--remote-mirror-backend", "huggingface",
    "--remote-mirror-target", "minkhoaa/langfens-rq1-cross-fold-checkpoints",
]
print("[PREFLIGHT] Executing checked command:")
print("  " + " ".join(preflight_cmd))
subprocess.run(preflight_cmd, check=True)

print("=" * 80)
print("[PREFLIGHT] Preflight verification PASSED successfully.")
print("[PREFLIGHT] Zero training, optimizer updates, or checkpoint generation occurred.")
print("=" * 80)
